In [1]:
import re
from konlpy.tag import Okt, Kkma
import pandas as pd

In [3]:
# 정규표현식 연습
raw_text = "안녕하세요!!! 123반갑습니다... @@@자연어 처리는 재미있어요 ^_^"
# 한글과 공백을 제외하고 모두 제거
cleaned_text = re.sub(r'[^가-힣\s]','',raw_text)
cleaned_text

'안녕하세요 반갑습니다 자연어 처리는 재미있어요 '

In [4]:
# 한국어 형태소 분석기 비교
okt = Okt()
kma = Kkma()
text = '나는 사과를 먹는다'
print(okt.pos(text))
print(kma.pos(text))

[('나', 'Noun'), ('는', 'Josa'), ('사과', 'Noun'), ('를', 'Josa'), ('먹는다', 'Verb')]
[('나', 'VV'), ('는', 'ETD'), ('사과', 'NNG'), ('를', 'JKO'), ('먹', 'VV'), ('는', 'EPT'), ('다', 'EFN')]


In [10]:
# okt 형태소 분석기를 이용 / 조사와 구두점 제거
text = '사과가 맛있다, 나는 사과를 먹는다'
pos = okt.pos(text)
filtered = [word for word, tag in pos if tag not in ('Josa', 'Punctuation')]
print('Okt POS:', pos)
print('조사/구두점 제거 결과:', filtered)

Okt POS: [('사과', 'Noun'), ('가', 'Josa'), ('맛있다', 'Adjective'), (',', 'Punctuation'), ('나', 'Noun'), ('는', 'Josa'), ('사과', 'Noun'), ('를', 'Josa'), ('먹는다', 'Verb')]
조사/구두점 제거 결과: ['사과', '맛있다', '나', '사과', '먹는다']


In [35]:
# 다음 영화 리뷰 데이터셋
# 1. 데이터 로드
# 2. 전처리 (정규식 이용해 한글과 공백만 추출)
# 3. 형태소 분석 및 품사 필터링 (명사 동사 형용사만 추출)
# 4. TTR 계산 (상위 100개 리뷰를 대상으로 전체 토큰수 대비 고유 타입의 수를 비율계산)
    # TTR < 0.5 어휘 반복이 많아서 TTR이 낮게 조정 else 다양한 어휘가 사용되어서 TTR이 높게 책정

import pandas as pd

df = pd.read_csv('daum_movie_review.csv')

df['cleaned_text'] = (
    df['review']
    .astype(str)
    .str.replace(r'[^가-힣\s]', '', regex=True)
)

from konlpy.tag import Okt
okt = Okt()
df['tokens'] = df['cleaned_text'].apply(
    lambda x: [
        word for word, tag in okt.pos(str(x), stem=True)
        if tag in ['Noun', 'Adjective', 'Verb']
    ]
)

df[['review', 'cleaned_text', 'tokens']].head()

,review,cleaned_text,tokens
0,돈 들인건 티가 나지만 보는 내내 하품만,돈 들인건 티가 나지만 보는 내내 하품만,"[돈, 들이다, 티, 나, 보다, 내내, 하품]"
1,몰입할수밖에 없다. 어렵게 생각할 필요없다. 내가 전투에 참여한듯 손에 땀이남.,몰입할수밖에 없다 어렵게 생각할 필요없다 내가 전투에 참여한듯 손에 땀이남,"[몰입, 하다, 없다, 어렵다, 생각, 하다, 필요없다, 내, 전투, 참여, 듯, ..."
2,이전 작품에 비해 더 화려하고 스케일도 커졌지만.... 전국 맛집의 음식들을 한데 ...,이전 작품에 비해 더 화려하고 스케일도 커졌지만 전국 맛집의 음식들을 한데 모은 것...,"[이전, 작품, 비다, 더, 화려하다, 스케일, 커지다, 전국, 맛집, 음식, 모으..."
3,이 정도면 볼만하다고 할 수 있음!,이 정도면 볼만하다고 할 수 있음,"[이, 정도, 볼, 하다, 하다, 수, 있다]"
4,재미있다,재미있다,[재미있다]


In [36]:
top_100 = df.head(100)

all_tokens = []

for tokens in top_100['tokens']:
    all_tokens.extend(tokens)

total_tokens = len(all_tokens)
unique_types = len(set(all_tokens))

ttr = unique_types / total_tokens if total_tokens > 0 else 0

print("전체 토큰 수:", total_tokens)
print("고유 타입 수:", unique_types)
print("TTR:", ttr)

if ttr < 0.5:
    print("어휘 반복이 많아서 TTR이 낮게 측정됨")
else:
    print("다양한 어휘가 사용되어 TTR이 높게 측정됨")

전체 토큰 수: 1243
고유 타입 수: 644
TTR: 0.5181013676588898
다양한 어휘가 사용되어 TTR이 높게 측정됨
